In [9]:
import numpy as np
import pandas as pd
import joblib
import nltk

In [10]:
df = pd.read_csv("../data/raw/movie_dataset.csv", sep=",")
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."


In [11]:
df.isna().sum()

id                            0
title                        19
vote_average                  0
vote_count                    0
status                        0
release_date             327002
revenue                       0
runtime                       0
adult                         0
backdrop_path           1086811
budget                        0
homepage                1295058
imdb_id                  769837
original_language             0
original_title               19
overview                 332847
popularity                    0
poster_path              517035
tagline                 1241582
genres                   638084
production_companies     831721
production_countries     702576
spoken_languages         674985
keywords                1088527
dtype: int64

In [12]:
df.shape

(1443026, 24)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1443026 entries, 0 to 1443025
Data columns (total 24 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1443026 non-null  int64  
 1   title                 1443007 non-null  str    
 2   vote_average          1443026 non-null  float64
 3   vote_count            1443026 non-null  int64  
 4   status                1443026 non-null  str    
 5   release_date          1116024 non-null  str    
 6   revenue               1443026 non-null  int64  
 7   runtime               1443026 non-null  int64  
 8   adult                 1443026 non-null  bool   
 9   backdrop_path         356215 non-null   str    
 10  budget                1443026 non-null  int64  
 11  homepage              147968 non-null   str    
 12  imdb_id               673189 non-null   str    
 13  original_language     1443026 non-null  str    
 14  original_title        1443007 non-null  str  

In [14]:
df.duplicated().value_counts()

False    1442645
True         381
Name: count, dtype: int64

In [15]:
df.drop_duplicates(subset='title', inplace=True)

In [16]:
# df[['backdrop_path', 'poster_path', 'homepage']]
df[['id','title','overview','tagline','genres','keywords','production_companies']].isna().sum()

id                            0
title                         1
overview                 299822
tagline                 1058890
genres                   546477
keywords                 918803
production_companies     698336
dtype: int64

In [17]:
df.dropna(subset=['title', 'poster_path'], inplace=True)

In [18]:
df[['id','title','overview','tagline','genres','keywords','production_companies']].isna().sum()

id                           0
title                        0
overview                147210
tagline                 654675
genres                  271197
keywords                535613
production_companies    366171
dtype: int64

In [19]:
df[df['overview'].isna()]

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
7636,42436,Natale in crociera,4.935,387,Released,2007-12-14,0,103,False,/iweTLbiGNv42fNcgQTmg19n0s0T.jpg,...,Natale in crociera,NaN,5.520,/n83LsfQCPReYembtNtuGwqUkUOS.jpg,NaN,Comedy,Filmauro,Italy,Italian,"cruise ship, cinepanettone"
10098,919952,Il grande giorno,6.276,252,Released,2022-12-22,0,90,False,/nujjJgEuMUaaslCPBFC4v3eoiAi.jpg,...,Il grande giorno,NaN,4.041,/fiJ8Qn6HmR3eupzbsR7Q2rMN2UN.jpg,NaN,Comedy,"Agidi Due, Medusa Film",Italy,"French, Italian",NaN
10928,17413,Incognito,5.507,224,Released,2009-04-28,0,94,False,/2Zn2IQmVW7MWO4oXJTu2oQzM2kT.jpg,...,Incognito,NaN,7.916,/wzgeFbUgDFSnVG63T0FrQOdLCc1.jpg,NaN,Comedy,"Same Player, Pathé, SCOPE Pictures, Weber Inve...",France,French,NaN
10971,154512,Lightning Strike,5.283,223,Released,2012-12-13,0,104,False,/ynoNkrU19q4yMcNXd2YEEGOyzcw.jpg,...,Colpi di fulmine,NaN,5.263,/o538evS1FqeftSKI7ixT2efkZKt.jpg,NaN,Comedy,Filmauro,Italy,Italian,"romantic comedy, cinepanettone"
11029,244514,Colpi di fortuna,4.700,221,Released,2013-12-21,0,0,False,/rwgxV7s75dri2MIs8s2ZedpTw0e.jpg,...,Colpi di fortuna,NaN,4.050,/rWumuliOwR6yAcP6ZVo0D0kjaGR.jpg,NaN,Comedy,Filmauro,Italy,Italian,"anthology, cinepanettone"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442891,918793,Milkstagram,0.000,0,Released,2021-11-11,0,0,False,NaN,...,젖스타그램,NaN,0.600,/lVzMw2n1XWedFwRHSY428tsh0RC.jpg,NaN,NaN,NaN,NaN,NaN,NaN
1442892,918680,德云社戊戌年封箱庆典,0.000,0,Released,2019-02-05,0,0,False,NaN,...,德云社戊戌年封箱庆典,NaN,0.600,/excW0LkmoXId0b6T4LqISR2KzGV.jpg,NaN,NaN,NaN,NaN,NaN,NaN
1442933,918515,德云社甲午年开箱庆典,0.000,0,Released,2014-02-05,0,0,False,NaN,...,德云社甲午年开箱庆典,NaN,0.600,/80B8nl3ZGAp3jauAc6WTgoNEDjb.jpg,NaN,NaN,NaN,NaN,NaN,NaN
1442972,918650,Searching for Mr. Moses,0.000,0,Released,1990-12-01,0,60,False,NaN,...,Auf der Suche nach Herrn Moses,NaN,0.600,/ejwG3axTsAD4YMwrccxd2M48Bfa.jpg,NaN,Documentary,NaN,Germany,German,NaN


In [20]:
selected_data = df[['id','title','overview','tagline','genres','keywords','production_companies']]

In [21]:
selected_data.head()

,id,title,overview,tagline,genres,keywords,production_companies
0,27205,Inception,"Cobb, a skilled thief who commits corporate es...",Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","rescue, mission, dream, airplane, paris, franc...","Legendary Pictures, Syncopy, Warner Bros. Pict..."
1,157336,Interstellar,The adventures of a group of explorers who mak...,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","rescue, future, spacecraft, race against time,...","Legendary Pictures, Syncopy, Lynda Obst Produc..."
2,155,The Dark Knight,Batman raises the stakes in his war on crime. ...,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","joker, sadism, chaos, secret identity, crime f...","DC Comics, Legendary Pictures, Syncopy, Isobel..."
3,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","future, society, culture clash, space travel, ...","Dune Entertainment, Lightstorm Entertainment, ..."
4,24428,The Avengers,When an unexpected enemy emerges and threatens...,Some assembly required.,"Science Fiction, Action, Adventure","new york city, superhero, shield, based on com...",Marvel Studios


In [22]:
cols = ['overview','tagline','genres','keywords','production_companies']

for i in cols:
    selected_data[i] = selected_data[i].fillna('')

In [23]:
def convert(obj):
    if isinstance(obj, str):
        return [i.strip() for i in obj.split(',')]
    return []

In [24]:
for col in cols:
    selected_data[col] = selected_data[col].apply(convert)

In [25]:
selected_data.head()

,id,title,overview,tagline,genres,keywords,production_companies
0,27205,Inception,"[Cobb, a skilled thief who commits corporate e...",[Your mind is the scene of the crime.],"[Action, Science Fiction, Adventure]","[rescue, mission, dream, airplane, paris, fran...","[Legendary Pictures, Syncopy, Warner Bros. Pic..."
1,157336,Interstellar,[The adventures of a group of explorers who ma...,[Mankind was born on Earth. It was never meant...,"[Adventure, Drama, Science Fiction]","[rescue, future, spacecraft, race against time...","[Legendary Pictures, Syncopy, Lynda Obst Produ..."
2,155,The Dark Knight,[Batman raises the stakes in his war on crime....,[Welcome to a world without rules.],"[Drama, Action, Crime, Thriller]","[joker, sadism, chaos, secret identity, crime ...","[DC Comics, Legendary Pictures, Syncopy, Isobe..."
3,19995,Avatar,"[In the 22nd century, a paraplegic Marine is d...",[Enter the world of Pandora.],"[Action, Adventure, Fantasy, Science Fiction]","[future, society, culture clash, space travel,...","[Dune Entertainment, Lightstorm Entertainment,..."
4,24428,The Avengers,[When an unexpected enemy emerges and threaten...,[Some assembly required.],"[Science Fiction, Action, Adventure]","[new york city, superhero, shield, based on co...",[Marvel Studios]


In [26]:
col = ['genres', 'keywords', 'production_companies']

for i in col:
    selected_data[i] = selected_data[i].apply(lambda x: [i.replace(" ", "") for i in x])

In [27]:
selected_data.head()

,id,title,overview,tagline,genres,keywords,production_companies
0,27205,Inception,"[Cobb, a skilled thief who commits corporate e...",[Your mind is the scene of the crime.],"[Action, ScienceFiction, Adventure]","[rescue, mission, dream, airplane, paris, fran...","[LegendaryPictures, Syncopy, WarnerBros.Pictures]"
1,157336,Interstellar,[The adventures of a group of explorers who ma...,[Mankind was born on Earth. It was never meant...,"[Adventure, Drama, ScienceFiction]","[rescue, future, spacecraft, raceagainsttime, ...","[LegendaryPictures, Syncopy, LyndaObstProducti..."
2,155,The Dark Knight,[Batman raises the stakes in his war on crime....,[Welcome to a world without rules.],"[Drama, Action, Crime, Thriller]","[joker, sadism, chaos, secretidentity, crimefi...","[DCComics, LegendaryPictures, Syncopy, IsobelG..."
3,19995,Avatar,"[In the 22nd century, a paraplegic Marine is d...",[Enter the world of Pandora.],"[Action, Adventure, Fantasy, ScienceFiction]","[future, society, cultureclash, spacetravel, s...","[DuneEntertainment, LightstormEntertainment, 2..."
4,24428,The Avengers,[When an unexpected enemy emerges and threaten...,[Some assembly required.],"[ScienceFiction, Action, Adventure]","[newyorkcity, superhero, shield, basedoncomic,...",[MarvelStudios]


In [28]:
selected_data['tags'] = (selected_data['overview'] + selected_data['genres'] + 
                         selected_data['keywords'] + selected_data['tagline'] + 
                          selected_data['production_companies'] 
                        )

In [29]:
selected_data.head()

,id,title,overview,tagline,genres,keywords,production_companies,tags
0,27205,Inception,"[Cobb, a skilled thief who commits corporate e...",[Your mind is the scene of the crime.],"[Action, ScienceFiction, Adventure]","[rescue, mission, dream, airplane, paris, fran...","[LegendaryPictures, Syncopy, WarnerBros.Pictures]","[Cobb, a skilled thief who commits corporate e..."
1,157336,Interstellar,[The adventures of a group of explorers who ma...,[Mankind was born on Earth. It was never meant...,"[Adventure, Drama, ScienceFiction]","[rescue, future, spacecraft, raceagainsttime, ...","[LegendaryPictures, Syncopy, LyndaObstProducti...",[The adventures of a group of explorers who ma...
2,155,The Dark Knight,[Batman raises the stakes in his war on crime....,[Welcome to a world without rules.],"[Drama, Action, Crime, Thriller]","[joker, sadism, chaos, secretidentity, crimefi...","[DCComics, LegendaryPictures, Syncopy, IsobelG...",[Batman raises the stakes in his war on crime....
3,19995,Avatar,"[In the 22nd century, a paraplegic Marine is d...",[Enter the world of Pandora.],"[Action, Adventure, Fantasy, ScienceFiction]","[future, society, cultureclash, spacetravel, s...","[DuneEntertainment, LightstormEntertainment, 2...","[In the 22nd century, a paraplegic Marine is d..."
4,24428,The Avengers,[When an unexpected enemy emerges and threaten...,[Some assembly required.],"[ScienceFiction, Action, Adventure]","[newyorkcity, superhero, shield, basedoncomic,...",[MarvelStudios],[When an unexpected enemy emerges and threaten...


In [30]:
new_df = selected_data[['id', 'title', 'tags']]

In [31]:
new_df.head()

,id,title,tags
0,27205,Inception,"[Cobb, a skilled thief who commits corporate e..."
1,157336,Interstellar,[The adventures of a group of explorers who ma...
2,155,The Dark Knight,[Batman raises the stakes in his war on crime....
3,19995,Avatar,"[In the 22nd century, a paraplegic Marine is d..."
4,24428,The Avengers,[When an unexpected enemy emerges and threaten...


In [32]:
new_df['tags'] = new_df['tags'].apply(lambda x : " ".join(x))

In [33]:
new_df.head()

,id,title,tags
0,27205,Inception,Cobb a skilled thief who commits corporate esp...
1,157336,Interstellar,The adventures of a group of explorers who mak...
2,155,The Dark Knight,Batman raises the stakes in his war on crime. ...
3,19995,Avatar,In the 22nd century a paraplegic Marine is dis...
4,24428,The Avengers,When an unexpected enemy emerges and threatens...


In [34]:
new_df['tags'][0]

'Cobb a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception" the implantation of another person\'s idea into a target\'s subconscious. Action ScienceFiction Adventure rescue mission dream airplane paris france virtualreality kidnapping philosophy spy allegory manipulation carcrash heist memory architecture losangeles california dreamworld subconscious Your mind is the scene of the crime. LegendaryPictures Syncopy WarnerBros.Pictures'

In [35]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [36]:
def stem(text):
    
    y = []
    
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)   

In [37]:
new_df['tags'] = new_df['tags'].apply(stem)

In [38]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vector = TfidfVectorizer(max_features= 5000, stop_words ='english')

In [39]:
vectors = vector.fit_transform(new_df['tags'])

In [40]:
vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14271059 stored elements and shape (789463, 5000)>

In [41]:
vector.get_feature_names_out()

array(['00', '000', '01', ..., 'zone', 'zoo', 'českátelev'],
      shape=(5000,), dtype=object)

In [42]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(vectors)

,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [43]:
# Index map (FAST LOOKUP)
index_map = pd.Series(new_df.index, index=new_df['title']).drop_duplicates()

In [44]:
def recommend(movie_name):
    
    movie_index = new_df[new_df['title'] == movie_name].index[0]
    
    distances, indices = model.kneighbors(vectors[movie_index], n_neighbors=10)
    
    for i in indices[0][1:]:
        print(new_df.iloc[i]['title'])

In [45]:
recommend('Spider-Man')

Homem-Aranha: Panóptico
Spider-Man: Homemade
Spider-Man 3
Spider-Man: MIRAGE
Man-spider
Spider-Man: Across the Spider-Verse: Across the Comics-Verse
Spider-Man: Across the Spider-Verse: Escape from Spider-Society
Spider-Man: Rise of The Green Goblin
Spider-Man: Across the Spider-Verse: Designing Spiders and Spots


In [46]:
import joblib

# movie metadata
joblib.dump(new_df, "../artifacts/model/movies.pkl")

# feature matrix
joblib.dump(vectors, "../artifacts/model/vectors.pkl")

# nearest neighbor model
joblib.dump(model, "../artifacts/model/nn_model.pkl")

joblib.dump(index_map, "../artifacts/model/index_map.pkl")

['../artifacts/model/index_map.pkl']